# Forecasting Customer-Care Ticket Volume (BigQuery ML style)

Companion notebook to `sql/forecast_bqml.sql`. The SQL file runs the real
ARIMA_PLUS model on BigQuery; this notebook reproduces the same idea locally
with `statsmodels` so the result is fully reproducible without cloud access,
and produces a forecast chart + MAPE for the portfolio.

Skill demonstrated: **Create ML Models with BigQuery ML** (Google Cloud badge).
Domain: predict next-month ticket volume so the care team can staff ahead.

In [ ]:
import os, sys
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

ROOT = os.path.abspath(os.path.join(os.get.dirname(__file__), '..'))
df = pd.read_csv(os.path.join(ROOT, 'data', 'tickets.csv'), parse_dates=['created_date'])
daily = df.groupby('created_date').size().asfreq('D').fillna(0)
train, test = daily[:-30], daily[-30:]
print(f'train={len(train)} days, test={len(test)} days')

In [ ]:
model = ARIMA(train, order=(2,1,2), seasonal_order=(1,1,1,7)).fit()
fc = model.forecast(steps=30)
mape = (abs(test - fc).mean() / test.mean()) * 100
print(f'30-day forecast MAPE: {mape:.1f}%')
print(f'Next 7 days: {fc.round(0).head(7).astype(int).tolist()}')

In [ ]:
plt.figure(figsize=(10,3.6))
plt.plot(train.index, train.values, label='Actual (train)')
plt.plot(test.index, test.values, label='Actual (test)')
plt.plot(test.index, fc.values, label=f'Forecast (MAPE {mape:.1f}%)', color='red')
plt.title('Daily Ticket Volume — ARIMA Forecast')
plt.legend(); plt.tight_layout()
out = os.path.join(ROOT, 'outputs', 'chart_forecast.png')
plt.savefig(out); print('saved', out)